# Praktikum Pertemuan 4 - Text Preprocessing II
Stemming, Normalisasi, dan perbandingan dua pipeline
NAMA : MUHAMMAD ADITYA RACHMAN
NPM : 2405040018

## 0. Persiapan

In [46]:
import re
import pandas as pd
from collections import Counter

In [47]:
df = pd.read_csv('dataset_ulasan_laptop_100.csv', keep_default_na=False, dtype=str)
teks = df['reviewContent'].tolist()     # daftar 100 ulasan

---
# BAGIAN 1 - PIPELINE A
Cleaning, Case Folding, Tokenisasi, Stopword

In [48]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stopword_a = set(StopWordRemoverFactory().get_stop_words()) 
print('Jumlah stopword:', len(stopword_a))

Jumlah stopword: 123


### A.1 Cleaning
Membuang link, tag HTML, tanda baca, simbol, dan spasi berlebih.

In [49]:
a_cleaning = []
for t in teks:
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)  
    t = re.sub(r'<[^>]+>', ' ', t)                 
    t = re.sub(r'[^A-Za-z0-9\s]', ' ', t)           
    t = re.sub(r'\s+', ' ', t).strip()              
    a_cleaning.append(t)

print('Sebelum:', teks[5])
print('Sesudah:', a_cleaning[5])

Sebelum: Barang udah sampai tapi pas dinyalain g bisa. Gmn pertanggung jawaban lazada?
Sesudah: Barang udah sampai tapi pas dinyalain g bisa Gmn pertanggung jawaban lazada


### A.2 Case Folding
Mengubah semua huruf menjadi huruf kecil.

In [50]:
a_lower = []
for t in a_cleaning:
    a_lower.append(t.lower())

print(a_lower[5])

barang udah sampai tapi pas dinyalain g bisa gmn pertanggung jawaban lazada


### A.3 Tokenisasi
Memotong kalimat menjadi kata-kata (token).

In [51]:
a_token = []
for t in a_lower:
    a_token.append(t.split())

print(a_token[5])
print('Total token:', sum(len(k) for k in a_token))

['barang', 'udah', 'sampai', 'tapi', 'pas', 'dinyalain', 'g', 'bisa', 'gmn', 'pertanggung', 'jawaban', 'lazada']
Total token: 1845


### A.4 Stopword
Membuang kata yang terlalu umum (dan, yang, di, dst).

In [52]:
a_hasil = []            
a_stop_dibuang = []     

for kalimat in a_token:
    simpan = []
    for kata in kalimat:
        if kata in stopword_a:
            a_stop_dibuang.append(kata)
        else:
            simpan.append(kata)
    a_hasil.append(simpan)

print('Sebelum:', a_token[5])
print('Sesudah:', a_hasil[5])
print()
print('Stopword terbuang :', len(a_stop_dibuang))
print('Token Pipeline A  :', sum(len(k) for k in a_hasil))

Sebelum: ['barang', 'udah', 'sampai', 'tapi', 'pas', 'dinyalain', 'g', 'bisa', 'gmn', 'pertanggung', 'jawaban', 'lazada']
Sesudah: ['barang', 'udah', 'pas', 'dinyalain', 'g', 'gmn', 'pertanggung', 'jawaban', 'lazada']

Stopword terbuang : 373
Token Pipeline A  : 1472


---
# BAGIAN 2 - PIPELINE B
Cleaning, Case Folding, Tokenisasi, Stopword, **Normalisasi, Stemming**

In [53]:
import csv
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

stopword_b = set(StopWordRemoverFactory().get_stop_words())
stemmer = StemmerFactory().create_stemmer()

# INI KAMUS GAUL DARI GITHUB
import os, urllib.request
file_kamus = 'colloquial-indonesian-lexicon.csv'
if not os.path.exists(file_kamus):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv',
        file_kamus)

kamus = {}
with open(file_kamus, encoding='utf-8', newline='') as f:
    for baris in csv.DictReader(f):
        gaul = baris['slang'].strip().lower()
        baku = baris['formal'].strip().lower()
        if gaul != baku and gaul not in kamus:
            kamus[gaul] = baku

print('Isi kamus:', len(kamus), 'pasangan kata')

Isi kamus: 4330 pasangan kata


### B.1 Cleaning

In [54]:
b_cleaning = []
for t in teks:
    t = re.sub(r'https?://\S+|www\.\S+', ' ', t)
    t = re.sub(r'<[^>]+>', ' ', t)
    t = re.sub(r'[^A-Za-z0-9\s]', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    b_cleaning.append(t)

print(b_cleaning[5])

Barang udah sampai tapi pas dinyalain g bisa Gmn pertanggung jawaban lazada


### B.2 Case Folding

In [55]:
b_lower = []
for t in b_cleaning:
    b_lower.append(t.lower())

print(b_lower[5])

barang udah sampai tapi pas dinyalain g bisa gmn pertanggung jawaban lazada


### B.3 Tokenisasi

In [56]:
b_token = []
for t in b_lower:
    b_token.append(t.split())

print(b_token[5])

['barang', 'udah', 'sampai', 'tapi', 'pas', 'dinyalain', 'g', 'bisa', 'gmn', 'pertanggung', 'jawaban', 'lazada']


### B.4 Stopword

In [57]:
b_stop = []             
b_stop_dibuang = []

for kalimat in b_token:
    simpan = []
    for kata in kalimat:
        if kata in stopword_b:
            b_stop_dibuang.append(kata)
        else:
            simpan.append(kata)
    b_stop.append(simpan)

print(b_stop[5])

['barang', 'udah', 'pas', 'dinyalain', 'g', 'gmn', 'pertanggung', 'jawaban', 'lazada']


### B.5 Normalisasi
normalisasi pake kamus gaul


In [58]:
b_norm = []     

for kalimat in b_stop:
    simpan = []
    for kata in kalimat:
        kata2 = re.sub(r'([a-z])\1{2,}', r'\1', kata)    
        if kata in kamus:
            for k in kamus[kata].split():
                simpan.append((kata, k))
        elif kata2 in kamus:
            for k in kamus[kata2].split():
                simpan.append((kata, k))
        else:
            simpan.append((kata, kata2))
    b_norm.append(simpan)

print('Sebelum:', b_stop[5])
print('Sesudah:', [hasil for asal, hasil in b_norm[5]])

Sebelum: ['barang', 'udah', 'pas', 'dinyalain', 'g', 'gmn', 'pertanggung', 'jawaban', 'lazada']
Sesudah: ['barang', 'sudah', 'pas', 'dinyalain', 'enggak', 'bagaimana', 'pertanggung', 'jawaban', 'lazada']


### B.6 Stemming
potong imbuhan pake sastrawi

In [59]:
b_hasil = []       
b_semua = []       
simpanan = {}       

for kalimat in b_norm:
    simpan = []
    for asal, norm in kalimat:
        if norm not in simpanan:
            simpanan[norm] = stemmer.stem(norm)
        dasar = simpanan[norm]
        b_semua.append((asal, norm, dasar))
        if dasar != '':
            simpan.append(dasar)
    b_hasil.append(simpan)

print('Sebelum:', [hasil for asal, hasil in b_norm[5]])
print('Sesudah:', b_hasil[5])
print()
print('Token Pipeline B:', sum(len(k) for k in b_hasil))

Sebelum: ['barang', 'sudah', 'pas', 'dinyalain', 'enggak', 'bagaimana', 'pertanggung', 'jawaban', 'lazada']
Sesudah: ['barang', 'sudah', 'pas', 'dinyalain', 'enggak', 'bagaimana', 'tanggung', 'jawab', 'lazada']

Token Pipeline B: 1473


---
# BAGIAN 3 - PERBANDINGAN A vs B

### 3.1 Jumlah token

In [60]:
token_a = sum(len(k) for k in a_hasil)
token_b = sum(len(k) for k in b_hasil)
kosakata_a = set(kata for kalimat in a_hasil for kata in kalimat)
kosakata_b = set(kata for kalimat in b_hasil for kata in kalimat)

pd.DataFrame({
    'Pipeline A': [token_a, len(kosakata_a), round(token_a / 100, 2)],
    'Pipeline B': [token_b, len(kosakata_b), round(token_b / 100, 2)],
}, index=['Jumlah token', 'Kata unik (kosakata)', 'Rata-rata token per ulasan'])

,Pipeline A,Pipeline B
Jumlah token,1472.00,1473.00
Kata unik (kosakata),667.00,571.00
Rata-rata token per ulasan,14.72,14.73


### 3.2 Kata yang berubah

In [61]:
tabel = pd.DataFrame(b_semua, columns=['asal', 'normal', 'stem'])

beda_norm  = tabel[tabel['asal'] != tabel['normal']]    # berubah karena normalisasi
beda_stem  = tabel[tabel['normal'] != tabel['stem']]    # berubah karena stemming
beda_total = tabel[tabel['asal'] != tabel['stem']]      # berubah di salah satunya

print('Token berubah (total)    :', len(beda_total), 'dari', token_a)
print('Berubah karena normalisasi:', len(beda_norm))
print('Berubah karena stemming   :', len(beda_stem))

Token berubah (total)    : 335 dari 1472
Berubah karena normalisasi: 141
Berubah karena stemming   : 212


In [62]:
# 10 perubahan normalisasi yang paling sering
beda_norm.groupby(['asal', 'normal']).size().sort_values(ascending=False).head(10).reset_index(name='jumlah')

,asal,normal,jumlah
0,yg,yang,11
1,add,ada,10
2,silahkan,silakan,10
3,ga,enggak,8
4,udah,sudah,8
5,rapih,rapi,6
6,bgt,banget,5
7,cepet,cepat,3
8,g,enggak,3
9,gak,enggak,3


### 3.3 Kata yang hilang

In [63]:
stop_hitung = Counter(a_stop_dibuang)
print('Stopword terbuang:', len(a_stop_dibuang), 'token')
pd.DataFrame(stop_hitung.most_common(10), columns=['stopword', 'jumlah'])

Stopword terbuang: 373 token


,stopword,jumlah
0,dan,33
1,dengan,25
2,yang,25
3,di,25
4,sudah,20
5,saya,19
6,ini,17
7,dari,14
8,sampai,13
9,ada,13


In [64]:
hitung_a = Counter(kata for kalimat in a_hasil for kata in kalimat)
hitung_b = Counter(kata for kalimat in b_hasil for kata in kalimat)

hilang = [k for k in hitung_a if k not in kosakata_b]      # ada di A, tidak ada di B
hilang = sorted(hilang, key=lambda k: -hitung_a[k])
baru   = [k for k in hitung_b if k not in kosakata_a]      # tidak ada di A, ada di B
muncul_lagi = {k: v for k, v in hitung_b.items() if k in stopword_a}

print('Kata di A yang tidak ada lagi di B:', len(hilang), 'dari', len(kosakata_a))
print('Kata baru di B                    :', len(baru))
print('Stopword muncul lagi di B         :', sum(muncul_lagi.values()), 'token')
print()
print('Contoh kata yang hilang:', hilang[:10])

Kata di A yang tidak ada lagi di B: 198 dari 667
Kata baru di B                    : 102
Stopword muncul lagi di B         : 69 token

Contoh kata yang hilang: ['pengiriman', 'yg', 'silahkan', 'add', 'udah', 'ga', 'diterima', 'memuaskan', 'rapih', 'barangnya']


### 3.4 Hasil stemming

In [65]:
beda_stem.groupby(['normal', 'stem']).size().sort_values(ascending=False).head(12).reset_index(name='jumlah')

,normal,stem,jumlah
0,pengiriman,kirim,14
1,silakan,sila,10
2,diterima,terima,7
3,semoga,moga,6
4,barangnya,barang,6
5,memuaskan,muas,6
6,memastikan,pasti,4
7,pokoknya,pokok,4
8,menjadi,jadi,4
9,laptopnya,laptop,4


### 3.5 Contoh kesalahan

In [66]:
def hitung(kolom1, nilai1, kolom2, nilai2):
    # menghitung berapa kali pasangan kata ini muncul
    return len(tabel[(tabel[kolom1] == nilai1) & (tabel[kolom2] == nilai2)])

kesalahan = pd.DataFrame([
    ('Stopword',    'tidak',        'dibuang', 'kata negasi hilang, makna terbalik', stop_hitung['tidak']),
    ('Normalisasi', 'add',          'ada',     'kata Inggris salah dikira singkatan', hitung('asal', 'add', 'normal', 'ada')),
    ('Normalisasi', 'd',            'ada',     'd artinya di',                        hitung('asal', 'd', 'normal', 'ada')),
    ('Normalisasi', 'it',           'itu',     'kata Inggris (grab it fast)',         hitung('asal', 'it', 'normal', 'itu')),
    ('Normalisasi', 'ga / g / gak', 'enggak',  'masih belum baku (harusnya tidak)',
        hitung('asal', 'ga', 'normal', 'enggak') + hitung('asal', 'g', 'normal', 'enggak') + hitung('asal', 'gak', 'normal', 'enggak')),
    ('Normalisasi', 'klo',          'kalo',    'masih belum baku (harusnya kalau)',   hitung('asal', 'klo', 'normal', 'kalo')),
    ('Stemming',    'silakan',      'sila',    'kata dasarnya terpotong',             hitung('normal', 'silakan', 'stem', 'sila')),
    ('Stemming',    'memuaskan',    'muas',    'harusnya puas',                       hitung('normal', 'memuaskan', 'stem', 'muas')),
    ('Stemming',    'pemesanan',    'mesan',   'harusnya pesan',                      hitung('normal', 'pemesanan', 'stem', 'mesan')),
    ('Stemming',    'tiki',         'tik',     'nama kurir dianggap berimbuhan',      hitung('normal', 'tiki', 'stem', 'tik')),
    ('Stemming',    'ternyata',     'nyata',   'artinya bergeser',                    hitung('normal', 'ternyata', 'stem', 'nyata')),
], columns=['tahap', 'kata asal', 'hasil', 'masalah', 'jumlah'])
kesalahan

,tahap,kata asal,hasil,masalah,jumlah
0,Stopword,tidak,dibuang,"kata negasi hilang, makna terbalik",12
1,Normalisasi,add,ada,kata Inggris salah dikira singkatan,10
2,Normalisasi,d,ada,d artinya di,2
3,Normalisasi,it,itu,kata Inggris (grab it fast),2
4,Normalisasi,ga / g / gak,enggak,masih belum baku (harusnya tidak),14
5,Normalisasi,klo,kalo,masih belum baku (harusnya kalau),2
6,Stemming,silakan,sila,kata dasarnya terpotong,10
7,Stemming,memuaskan,muas,harusnya puas,6
8,Stemming,pemesanan,mesan,harusnya pesan,1
9,Stemming,tiki,tik,nama kurir dianggap berimbuhan,1


In [67]:
# Contoh paling parah: kata "tidak" dibuang, artinya jadi terbalik
print('ASLI:', teks[7])
print('A   :', ' '.join(a_hasil[7]))
print('B   :', ' '.join(b_hasil[7]))

ASLI: Kenapa chargernya tidak lengkap....?? G bisa langsung dipakai buat charger..., kita harus nyari dan beli lagi sambungan buat chargernya.., tolong d bantu
A   : chargernya lengkap g langsung dipakai buat charger nyari beli sambungan buat chargernya d bantu
B   : chargernya lengkap enggak langsung pakai buat charger cari beli sambung buat chargernya ada bantu


---
# BAGIAN 4 - SASTRAWI vs NLTK

nltk gak punya stemmer bahasa indonesia jadi cuma membandingkan stopword sama steeming

In [78]:
import nltk
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')            

from nltk.corpus import stopwords
stopword_nltk = set(stopwords.words('indonesian'))
stopword_sastrawi = stopword_a

print('Stopword Sastrawi:', len(stopword_sastrawi), 'kata')
print('Stopword NLTK    :', len(stopword_nltk), 'kata')

Stopword Sastrawi: 123 kata
Stopword NLTK    : 757 kata


### 4.1 Stopword

In [79]:
semua_kata = [kata for kalimat in a_token for kata in kalimat]    
buang_sastrawi = sum(1 for k in semua_kata if k in stopword_sastrawi)
buang_nltk     = sum(1 for k in semua_kata if k in stopword_nltk)

print('Jumlah kata awal  :', len(semua_kata))
print(f'Dibuang Sastrawi  : {buang_sastrawi} ({buang_sastrawi / len(semua_kata):.1%})')
print(f'Dibuang NLTK      : {buang_nltk} ({buang_nltk / len(semua_kata):.1%})')

Jumlah kata awal  : 1845
Dibuang Sastrawi  : 373 (20.2%)
Dibuang NLTK      : 544 (29.5%)


In [80]:
negasi = ['tidak', 'bukan', 'belum', 'tanpa', 'jangan', 'kurang', 'tak', 'kecuali']
pd.DataFrame({
    'Sastrawi': [k in stopword_sastrawi for k in negasi],
    'NLTK':     [k in stopword_nltk for k in negasi],
}, index=negasi).T

,tidak,bukan,belum,tanpa,jangan,kurang,tak,kecuali
Sastrawi,True,False,True,True,False,False,False,True
NLTK,True,True,True,True,True,True,True,False


In [81]:
hitung_semua = Counter(semua_kata)
hanya_nltk = {k: v for k, v in hitung_semua.items() if k in stopword_nltk and k not in stopword_sastrawi}
sorted(hanya_nltk.items(), key=lambda x: -x[1])[:10]

[('lebih', 11),
 ('hari', 8),
 ('buat', 8),
 ('jadi', 7),
 ('baik', 7),
 ('datang', 5),
 ('sangat', 5),
 ('sekali', 5),
 ('memastikan', 4),
 ('banyak', 4)]

### 4.2 Stemming

In [82]:
from nltk.stem import PorterStemmer, SnowballStemmer

print('Bahasa yang didukung Snowball NLTK:', SnowballStemmer.languages)
print('Ada Bahasa Indonesia?', 'indonesian' in SnowballStemmer.languages)

porter = PorterStemmer()     # stemmer pake porter

Bahasa yang didukung Snowball NLTK: ('arabic', 'danish', 'dutch', 'english', 'finnish', 'french', 'german', 'hungarian', 'italian', 'norwegian', 'porter', 'portuguese', 'romanian', 'russian', 'spanish', 'swedish')
Ada Bahasa Indonesia? False


In [73]:
contoh = ['pengiriman', 'diterima', 'barangnya', 'memuaskan', 'pelayanan',
          'bagus', 'puas', 'mulus', 'packing', 'windows']

pd.DataFrame({
    'kata': contoh,
    'Sastrawi': [stemmer.stem(k) for k in contoh],
    'NLTK Porter': [porter.stem(k) for k in contoh],
})

,kata,Sastrawi,NLTK Porter
0,pengiriman,kirim,pengiriman
1,diterima,terima,diterima
2,barangnya,barang,barangnya
3,memuaskan,muas,memuaskan
4,pelayanan,layan,pelayanan
5,bagus,bagus,bagu
6,puas,puas,pua
7,mulus,mulus,mulu
8,packing,packing,pack
9,windows,windows,window


In [83]:
kata_unik = set(tabel['normal'])
ubah_sastrawi = sum(1 for k in kata_unik if stemmer.stem(k) != k)
ubah_porter   = sum(1 for k in kata_unik if porter.stem(k) != k)

print('Kata unik            :', len(kata_unik))
print('Diubah Sastrawi      :', ubah_sastrawi)
print('Diubah NLTK Porter   :', ubah_porter)

Kata unik            : 632
Diubah Sastrawi      : 135
Diubah NLTK Porter   : 76


### Kesimpulan Sastrawi vs NLTK

- **Stemming:** Sastrawi paham imbuhan Indonesia (`pengiriman` menjadi `kirim`, `barangnya` menjadi `barang`). NLTK tidak punya stemmer Indonesia, dan Porter (Inggris) tidak mengubah kata berimbuhan itu, malah merusak kata dasar (`bagus` menjadi `bagu`, `puas` menjadi `pua`, `mulus` menjadi `mulu`). Sastrawi juga tidak sempurna (`silakan` menjadi `sila`), tetapi jauh lebih cocok.
- **Stopword:** daftar NLTK jauh lebih banyak (757 kata dibanding 123) dan membuang lebih banyak kata, termasuk kata yang bermakna seperti `baik`, `sangat`, `sekali`, dan `hari`. NLTK juga membuang lebih banyak kata negasi. Daftar Sastrawi lebih aman untuk ulasan.
- **Keduanya** membuang kata `tidak`, sehingga "tidak lengkap" menjadi "lengkap". Kata negasi sebaiknya dikeluarkan dari daftar stopword.
- **Saran:** pakai Sastrawi untuk stemming, dan pakai daftar stopword Sastrawi dengan kata negasi dipertahankan.